# Przewidywanie Ryzyka Kredytowego - Model Bazowy

## 1. Kontekst Biznesowy i Cel Projektu
Celem tego notatnika jest zbudowanie modelu uczenia maszynowego, który na podstawie danych demograficznych i finansowych klienta oceni prawdopodobieństwo niespłacenia przez niego kredytu.

Rozwiązujemy tu problem klasyfikacji binarnej , w którym nasza zmienna docelowa (`TARGET`) przyjmuje wartości:
* **0** - klient spłaca kredyt terminowo
* **1** - klient ma problemy ze spłatą

Zjawisko niewypłacalności występuje rzadko (8%), co stanowi główne wyzwanie analityczne tego projektu (niezbalansowane klasy).

## 2. Plan Działania
Aby zbudować skuteczny i powtarzalny proces, zrobię następujące kroki:
1. **Środowisko i Dane:** Wczytanie historycznych danych aplikacyjnych z platformy Kaggle (`application_train.csv`).
2. **Preprocessing:** Przygotowanie danych dla algorytmów uczenia maszynowego (obsługa braków danych).
3. **Zbalansowany Podział:** Wydzielenie zbioru treningowego i testowego z zachowaniem oryginalnych proporcji.
4. **Trening Modelu:** Użycie algorytmu Random Forest z parametrami korygującymi niezbalansowanie klas.
5. **Ewaluacja (Weryfikacja):** Ocena modelu za pomocą macierzy błędów, metryki ROC-AUC oraz identyfikacja najważniejszych czynników ryzyka.



In [18]:
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve



### Etap 1: Inicjalizacja Środowiska i Wczytanie Danych
Rozpoczynamy od zaimportowania niezbędnych bibliotek do manipulacji danymi (`pandas`), uczenia maszynowego (`scikit-learn`) oraz wizualizacji (`matplotlib`, `seaborn`). Następnie podpinamy Dysk Google w celu wczytania surowego pliku CSV.


In [19]:
drive.mount('/content/drive')

sciezka_do_pliku = '/content/drive/MyDrive/application_train.csv'
df = pd.read_csv(sciezka_do_pliku)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0




### Etap 2: Preprocessing i Podział Danych (Train / Test Split)

Surowe dane aplikacyjne nie mogą być bezpośrednio "zrozumiane" przez model matematyczny. Musimy wykonać dwa niezbędne kroki transformacji:

1. **One-Hot Encoding (`pd.get_dummies`):** Algorytmy ML wymagają danych liczbowych. Zamieniamy wszystkie zmienne tekstowe (kategoryczne, np. płeć czy wykształcenie) na zera lub jedynki.
2. **Obsługa Braków Danych (`fillna`):** Występowanie pustych komórek (NaN) powoduje błędy w bibliotece `scikit-learn`. Na potrzeby modelu bazowego, zastępujemy wszystkie braki wartością `0`.

**Podział z zachowaniem stratyfikacji:**
Na koniec dzielimy zbiór na dane treningowe (80%) oraz testowe (20%). Użycie parametru `stratify=y` jest tutaj kluczowe – gwarantuje nam to, że w obu podzbiorach zachowamy dokładnie taki sam odsetek (8%) osób niespłacających kredytu, co w oryginalnych danych.

In [20]:
df_encoded = pd.get_dummies(df)
df_encoded = df_encoded.fillna(0)


X = df_encoded.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_encoded['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\n--- PODSUMOWANIE ---")
print(f"Wymiary danych treningowych (X_train): {X_train.shape}")
print(f"Wymiary danych testowych (X_test): {X_test.shape}")
print(f"Procent złych kredytów w treningowym: {y_train.mean():.2%}")
print(f"Procent złych kredytów w testowym: {y_test.mean():.2%}")



--- PODSUMOWANIE ---
Wymiary danych treningowych (X_train): (246008, 244)
Wymiary danych testowych (X_test): (61503, 244)
Procent złych kredytów w treningowym: 8.07%
Procent złych kredytów w testowym: 8.07%


### Etap 3: Budowa i Trenowanie Modelu

Na tym etapie inicjujemy i uczymy nasz algorytm, wykorzystując przygotowane wcześniej dane.

1. **Konfiguracja Lasu Losowego:** Parametr `class_weight='balanced'` jest tutaj kluczowy – sztucznie nadaje wyższą wagę mniejszościowej klasie (osobom niespłacającym kredytu), zapobiegając jej ignorowaniu przez model. Ustawienie `max_depth=10` chroni przed overfittingiem, a `n_jobs=-1` maksymalnie przyspiesza obliczenia, wykorzystując wszystkie dostępne rdzenie procesora.
2. **Trenowanie:** Model "uczy się" zależności w danych na podstawie przygotowanego zbioru treningowego.
3. **Predykcje:** Algorytm przewiduje wyniki dla niewidzianego wcześniej zbioru testowego, zwracając zarówn

In [ ]:

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Model skończył się uczyć. Robimy predykcje na zbiorze testowym.")

y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]


### Etap 4: Ewaluacja i Wizualizacja Wyników

Po wytrenowaniu modelu przechodzimy do oceny jego skuteczności w praktyce, generując odpowiednie zestawienia i wykresy.

1. **Macierz Błędów:** Pierwszy wykres pozwala wizualnie ocenić, ile przypadków zaklasyfikowaliśmy bezbłędnie, a gdzie model się pomylił (np. oznaczając kogoś jako spłacającego, podczas gdy w rzeczywistości przestał płacić).
2. **Krzywa ROC i metryka AUC:** Środkowy wykres obrazuje zdolność algorytmu do poprawnego rozróżniania obu klas. Metryka ROC-AUC to pojedyncza liczba określająca ogólną jakość modelu – im bliżej wartości 1.0, tym algorytm radzi sobie lepiej.
3. **Najważniejsze Cechy:** Trzeci wykres (słupkowy) wyciąga z modelu TOP 10 zmiennych, które miały najsilniejszy wpływ na proces decyzyjny algorytmu.
4. **Raport Klasyfikacji:** Wyświetlenie w konsoli tekstowego podsumowania najważniejszych twardych metryk dl



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Macierz Błędów')
axes[0].set_xlabel('Przewidziane przez model')
axes[0].set_ylabel('Rzeczywiste')

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (obszar = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_title('Krzywa ROC')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc="lower right")

importances = rf_model.feature_importances_
# 10 cech
indices = importances.argsort()[-10:]

axes[2].barh(range(len(indices)), importances[indices], align='center', color='skyblue')
axes[2].set_yticks(range(len(indices)))
axes[2].set_yticklabels([X.columns[i] for i in indices])
axes[2].set_title('Top 10 Najważniejszych Cech')

plt.tight_layout()
plt.show()

print("\n--- RAPORT KLASYFIKACJI ---")
print(classification_report(y_test, y_pred))
print(f"Metryka ROC-AUC: {roc_auc:.4f}")
